# Retail Business Performance — Exploratory Data Analysis

Exploratory analysis on the cleaned Superstore dataset, ahead of building the Power BI dashboard.
Goal: confirm the patterns that inform the Executive Summary, Product Analysis, and Ad Hoc Analysis dashboard pages.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('../data/cleaned/superstore_cleaned_python.csv', parse_dates=['Order Date', 'Ship Date'])
df.shape

## 1. Quick overview

In [ ]:
df.head()

In [ ]:
df[['Sales', 'Profit', 'Discount', 'Quantity', 'Profit Margin']].describe()

## 2. Sales and profit trend over time

In [ ]:
monthly = df.set_index('Order Date').resample('M')[['Sales', 'Profit']].sum()

fig, ax = plt.subplots(figsize=(12, 5))
monthly['Sales'].plot(ax=ax, label='Sales')
monthly['Profit'].plot(ax=ax, label='Profit')
ax.set_title('Monthly Sales and Profit')
ax.set_ylabel('Amount')
ax.legend()
plt.show()

## 3. Sales and profit by region

In [ ]:
region_summary = df.groupby('Region')[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)
region_summary['Profit Margin %'] = (region_summary['Profit'] / region_summary['Sales']) * 100
region_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
region_summary['Sales'].plot(kind='bar', ax=ax, color='#1f77b4')
ax.set_title('Total Sales by Region')
ax.set_ylabel('Sales')
plt.xticks(rotation=0)
plt.show()

## 4. Sales and profit by category / sub-category

In [ ]:
category_summary = df.groupby('Category')[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)
category_summary

In [ ]:
subcat_summary = df.groupby('Sub-Category')[['Sales', 'Profit']].sum().sort_values('Profit')
subcat_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if v < 0 else '#2ca02c' for v in subcat_summary['Profit']]
subcat_summary['Profit'].plot(kind='barh', ax=ax, color=colors)
ax.set_title('Total Profit by Sub-Category (red = net loss)')
ax.set_xlabel('Profit')
plt.show()

## 5. Discount impact on profit

Confirms the finding used on the Ad Hoc Analysis dashboard page: profit drops sharply as discount level increases, and Bookcases is the standout sub-category running a net loss.

In [ ]:
band_order = ['No Discount', 'Low (1-20%)', 'Medium (21-40%)', 'High (40%+)']
discount_band_summary = df.groupby('Discount Band')['Profit'].sum().reindex(band_order)
discount_band_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
discount_band_summary.plot(kind='bar', ax=ax, color='#1f77b4')
ax.set_title('Total Profit by Discount Band')
ax.set_ylabel('Profit')
plt.xticks(rotation=20)
plt.show()

In [ ]:
discount_by_subcat = df.groupby('Sub-Category').agg(
    avg_discount=('Discount', 'mean'),
    total_profit=('Profit', 'sum')
).sort_values('total_profit')

discount_by_subcat.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(discount_by_subcat['avg_discount'], discount_by_subcat['total_profit'])
for name, row in discount_by_subcat.iterrows():
    ax.annotate(name, (row['avg_discount'], row['total_profit']), fontsize=8, alpha=0.7)
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Average Discount')
ax.set_ylabel('Total Profit')
ax.set_title('Average Discount vs Total Profit by Sub-Category')
plt.show()

## 6. Correlation check

In [ ]:
corr_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation: Sales, Quantity, Discount, Profit')
plt.show()

## 7. Summary of findings

- West and East regions drive the majority of sales; South consistently underperforms.
- Category mix is fairly balanced: Technology (36.4%), Furniture (32.3%), Office Supplies (31.3%).
- Discount and Profit are negatively correlated — heavier discounting erodes margin.
- Bookcases is the only sub-category with a net loss overall, tied to a comparatively high average discount (~21%).
- Binders carries the highest average discount (~37%) across sub-categories and remains profitable, but is worth monitoring.

These findings are carried into the Power BI dashboard (Executive Summary, Product Analysis, and Ad Hoc Analysis pages) and the written insights memo.